# Common Data Pipeline (Work in Progress)

This is an initial version notebook of the common data  pipeline.  
It will be updated after team discussion.

In [1]:
# Import libraries

import json
import pandas as pd

our task is object detection, so we need bounding boxes. The image_annotations.json file does not have bounding boxes. The bounding boxes coordinates are in **panoptic_annotations.json file.**(each object has a category_id and bbox) but these informations are not available in image_annotations.json file.

In [2]:
# Load panoptic annotation file

def load_panoptic(split):
    with open(f"dataset/lars_v1.0.0_annotations/{split}/panoptic_annotations.json", "r", encoding="utf-8") as f:
        return json.load(f)

In [3]:
# Load train and val data

train_data = load_panoptic("train")
val_data = load_panoptic("val")
print("Train images:", len(train_data["images"]))
print("Val images:", len(val_data["images"]))


Train images: 2605
Val images: 198


The original test split has no labels.  
So, we cannot use it for evaluation.  

We will train the models on 2300 training images.  
We will test the models on 300 images taken from the training data.

In [4]:
# Select only object classes (isthing = 1)

thing_categories = {
    cat["id"]: cat["name"]
    for cat in train_data["categories"]
    if cat["isthing"] == 1
}

thing_categories


{11: 'Boat/ship',
 12: 'Row boats',
 13: 'Paddle board',
 14: 'Buoy',
 15: 'Swimmer',
 16: 'Animal',
 17: 'Float',
 19: 'Other'}

In [5]:
# Create image records (all images)

def build_image_records(data, split):
    records = []

    for img in data["images"]:
        records.append({
            "split": split,
            "file_name": img["file_name"],
            "width": img["width"],
            "height": img["height"]
        })

    return records

In [6]:
# Create annotation records (only objects with bbox)

def build_annotation_records(data, split, thing_categories):
    image_id_to_info = {img["id"]: img for img in data["images"]}
    records = []

    for ann in data["annotations"]:
        img_info = image_id_to_info[ann["image_id"]]

        for seg in ann["segments_info"]:
            if seg["category_id"] in thing_categories:
                records.append({
                    "split": split,
                    "file_name": img_info["file_name"],
                    "width": img_info["width"],
                    "height": img_info["height"],
                    "category_id": seg["category_id"],
                    "category_name": thing_categories[seg["category_id"]],
                    "bbox": seg["bbox"]
                })

    return records

In [7]:
# Build image dataframe

train_images = build_image_records(train_data, "train")
val_images = build_image_records(val_data, "val")

images_df = pd.DataFrame(train_images + val_images)

print("Total images:", len(images_df))
images_df.head()


Total images: 2803


,split,file_name,width,height
0,train,mastr_0001_00009.jpg,1278,958
1,train,mastr_0002_00009.jpg,1278,958
2,train,mastr_0003_00009.jpg,1278,958
3,train,mastr_0004_00009.jpg,1278,958
4,train,mastr_0005_00009.jpg,1278,958


In [8]:
# Build annotation dataframe

train_annotations = build_annotation_records(train_data, "train", thing_categories)
val_annotations = build_annotation_records(val_data, "val", thing_categories)

annotations_df = pd.DataFrame(train_annotations + val_annotations)

print("Total annotations:", len(annotations_df))
annotations_df.head()

Total annotations: 10428


,split,file_name,width,height,category_id,category_name,bbox
0,train,mastr_0003_00009.jpg,1278,958,11,Boat/ship,"[0, 725, 52, 143]"
1,train,mastr_0003_00009.jpg,1278,958,19,Other,"[0, 544, 20, 58]"
2,train,mastr_0003_00009.jpg,1278,958,14,Buoy,"[199, 575, 10, 8]"
3,train,mastr_0004_00009.jpg,1278,958,14,Buoy,"[289, 534, 51, 78]"
4,train,mastr_0004_00009.jpg,1278,958,14,Buoy,"[589, 566, 6, 8]"


In [9]:
# Show the number of different object classes

print(annotations_df["category_name"].nunique())

# Show all class names

print(annotations_df["category_name"].unique())

# Count the number of objects in each class

print(annotations_df["category_name"].value_counts())

8
['Boat/ship' 'Other' 'Buoy' 'Float' 'Row boats' 'Swimmer' 'Animal'
 'Paddle board']
category_name
Boat/ship       6707
Buoy            1693
Other            574
Row boats        478
Animal           391
Swimmer          375
Paddle board     184
Float             26
Name: count, dtype: int64


In [10]:
# Check the number of different image widths and heights

print("Unique widths:", images_df["width"].nunique())
print("Unique heights:", images_df["height"].nunique())

Unique widths: 5
Unique heights: 10


In [11]:
# Show all different image sizes

images_df[["width", "height"]].drop_duplicates().sort_values(["width", "height"]).reset_index(drop=True)

,width,height
0,640,400
1,640,480
2,1278,958
3,1280,676
4,1280,702
5,1280,720
6,1280,1024
7,1920,1080
8,1920,1440
9,2208,1242


Images in the dataset have different sizes.  
During training, they are resized to the same input size.  
4 model should use the same image size for fair comparison.


In [12]:
# Count each image size

images_df.groupby(["width", "height"]).size().reset_index(name="count").sort_values("count", ascending=False).reset_index(drop=True)

,width,height,count
0,1278,958,1323
1,1920,1080,588
2,1280,720,517
3,2208,1242,155
4,640,480,82
5,1280,1024,64
6,640,400,60
7,1280,676,7
8,1920,1440,4
9,1280,702,3


The images in the dataset have high resolution and also contain small objects.  
For this reason, we can choose 768 × 768 as a good starting input size.  maybe for other experiments we can try 960 ×  960 later. (letterbox resizing )
When images are resized and padded, **bounding boxes must also be updated.**

Common Augmentation : 

Horizontal flip, p = 0.5
Light brightness / contrast change, p = 0.3
Light color jitter (HSV or similar), p = 0.3
Small scale change, p = 0.2–0.3
Small translation, p = 0.2
Small rotation only (about ±5°), p = 0.2


also we should implement same training setup epochs, batch size, learning rate etc. and we should **normalize de images same scale before giving them to the models.**


In [13]:
# Save CSV files

images_df.to_csv("common_images.csv", index=False)
annotations_df.to_csv("common_annotations.csv", index=False)


These files will be used by all models for training and evaluation.